# 🎙️ VoiceStudio on Google Colab (free GPU)

Voice cloning, voice design, dubbing aur TTS — PC off hone ke baad bhi, Colab ke free GPU pe.

**Pehle ye karo:** upar menu mein **Runtime → Change runtime type → T4 GPU** select karo, phir **Save**.

Phir **Runtime → Run all** dabao. 8–10 min lagega (pehli baar install hota hai). Last cell mein link aayega — **wahi link khol lo** (mobile pe bhi chalta hai).

> Free Colab session 2–4 ghante baad band ho jata hai, aur har nayi session mein model dobara download hota hai. Browser tab band mat karo jab tak use kar rahe ho.

## 1. GPU check

In [ ]:
# ── 1. GPU check ─────────────────────────────────────────────────────────────
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    print("✅ GPU mil gaya — chalu rakho.")
else:
    print("=" * 70)
    print("❌ GPU nahi hai. Upar: Runtime → Change runtime type → T4 GPU → Save,")
    print("   phir Runtime → Run all dobara chalao.")
    print("   (CPU pe bhi chal jayega, par bahut slow hoga.)")
    print("=" * 70)

## 2. Install (5–8 min, sirf pehli baar)

In [ ]:
# ── 2. Clone + install ───────────────────────────────────────────────────────
# Same steps as the official VoiceStudio Colab notebook (notebooks/OmniVoice_Studio_Colab.ipynb),
# pointed at Moneyinvest700/VoiceStudio. Safe to re-run.
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/Moneyinvest700/VoiceStudio"
REPO_DIR = "/content/VoiceStudio"
BUN = "/root/.bun/bin/bun"
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}", flush=True)
    rc = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode
    if rc != 0:
        raise SystemExit(f"\n❌ FAILED: {what or shown} (exit {rc}). Upar error dekho; ye cell dobara chalana safe hai.")

# 2a. Repo
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo pehle se hai: {REPO_DIR}")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], what="git clone")

# 2b. System packages
run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1 lsof", what="apt-get install")

# 2c. bun (web UI build ke liye)
if not os.path.exists(BUN):
    run("curl -fsSL https://bun.sh/install | bash", what="bun installer")
os.environ["PATH"] = os.path.dirname(BUN) + os.pathsep + os.environ["PATH"]
run([BUN, "--version"], what="bun version")

# 2d. Web UI build (frontend/dist) — ~2 min, skipped if already built
dist_index = os.path.join(REPO_DIR, "frontend", "dist", "index.html")
if os.path.exists(dist_index):
    print("Web UI pehle se built hai — skip.")
else:
    run([BUN, "install", "--frozen-lockfile"], cwd=REPO_DIR, what="bun install")
    run([BUN, "run", "--cwd", "electron", "build:web"], cwd=REPO_DIR, what="web UI build")
    if not os.path.exists(dist_index):
        raise SystemExit("❌ Web UI build hua par frontend/dist/index.html nahi mila — upar vite errors dekho.")

# 2e. Python backend deps (same as the official Docker image / Colab notebook)
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
run(["uv", "pip", "install", "--system", "--no-cache",
     "--constraint", "deploy/torch-constraints.txt", "."],
    cwd=REPO_DIR, what="backend install")

# 2f. cuDNN 8 compat for WhisperX (transcription on GPU); non-fatal
os.makedirs(os.path.join(REPO_DIR, ".venv"), exist_ok=True)
subprocess.run([sys.executable, os.path.join(REPO_DIR, "scripts", "setup.py")])

# 2g. Import check in a fresh interpreter
run([sys.executable, "-c",
     "import torch, torchaudio, uvicorn, fastapi, transformers; "
     "print(f'torch {torch.__version__}, torchaudio {torchaudio.__version__}, "
     "transformers {transformers.__version__}, CUDA: {torch.cuda.is_available()}'); "
     "from omnivoice.models.omnivoice import OmniVoice; "
     "from transformers import HiggsAudioV2TokenizerModel; "
     "print('✅ Install OK')"],
    cwd=REPO_DIR, what="import check")

## 3. Model download (~2.3 GB, 1–3 min)

In [ ]:
# ── 3. Default voice model (k2-fsa/OmniVoice) pehle se download kar lo ───────
# Nahi karoge to pehli generation pe download hoga aur 300s timeout aa sakta hai.
from huggingface_hub import snapshot_download
path = snapshot_download("k2-fsa/OmniVoice")
print("✅ Model ready:", path)

## 4. Launch + link 🔗

In [ ]:
# ── 4. Backend start karo, phir public link banao ────────────────────────────
import json, os, re, subprocess, sys, time, urllib.request

REPO_DIR = "/content/VoiceStudio"
PORT = 3900
LOG_PATH = "/content/voicestudio_backend.log"
CF_LOG = "/content/cloudflared.log"

def health():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = health()
if info:
    print("Backend pehle se chal raha hai:", info)
else:
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    time.sleep(1)
    env = os.environ.copy()
    env["OMNIVOICE_SERVER_MODE"] = "1"                 # headless server mode (same as Docker image)
    env["OMNIVOICE_DATA_DIR"] = "/content/voicestudio_data"
    env["OMNIVOICE_GENERATE_TIMEOUT_S"] = "900"        # first generation on T4 can be slow
    env["PYTHONUNBUFFERED"] = "1"
    log = open(LOG_PATH, "ab")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--app-dir", "backend",
         "--host", "127.0.0.1", "--port", str(PORT)],
        cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)
    print(f"Backend start ho raha hai (PID {proc.pid})", end="", flush=True)
    deadline = time.time() + 300
    while time.time() < deadline and proc.poll() is None:
        info = health()
        if info:
            break
        print(".", end="", flush=True)
        time.sleep(3)
    print()
    if not info:
        tail = "".join(open(LOG_PATH, errors="replace").readlines()[-40:])
        raise SystemExit(f"❌ Backend 5 min mein up nahi hua.\n--- log ---\n{tail}\n--- end ---\nCell 2 dobara chalao, phir ye cell.")
    print("✅ Backend up:", info)
    if "cuda" not in str(info.get("device", "")):
        print("⚠️ GPU use nahi ho raha (device:", info.get("device"), ") — cell 1 dekho.")

# Public URL via Cloudflare quick tunnel (koi account nahi chahiye)
CF = "/usr/local/bin/cloudflared"
if not os.path.exists(CF):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", CF], check=True)
    os.chmod(CF, 0o755)

url = None
for _attempt in range(3):
    subprocess.run("pkill -f 'cloudflared tunnel' 2>/dev/null || true", shell=True)
    time.sleep(1)
    cf_log = open(CF_LOG, "wb")
    subprocess.Popen([CF, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                     stdout=cf_log, stderr=subprocess.STDOUT)
    for _ in range(30):
        time.sleep(2)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(CF_LOG, errors="replace").read())
        if m:
            url = m.group(0)
            break
    if url:
        break

print()
print("=" * 70)
if url:
    print("🔗 VoiceStudio LINK (ye khol lo, mobile pe bhi chalega):")
    print()
    print("   ", url)
    print()
    print("Ye link sirf is Colab session tak chalega. Kisi ko mat bhejna —")
    print("jiske paas link hai wo app use kar sakta hai.")
else:
    print("⚠️ Cloudflare tunnel nahi bana. Niche wala Colab link use karo (Google login wala):")
print("=" * 70)

# Backup: Colab ka apna private link (sirf tumhare Google account mein khulta hai)
try:
    from google.colab import output
    print("\nColab private link (backup):")
    output.serve_kernel_port_as_window(PORT)
except Exception as e:
    print("(Colab port proxy unavailable:", e, ")")

## Session zinda rakhne ke liye
Ye cell chalta rehne do — ye bas backend ko dekhta rehta hai. Band karna ho to **Runtime → Disconnect and delete runtime**.

**Problem aaye to:** Runtime → Restart session, phir Run all dobara. Generated audio `/content/voicestudio_data` mein hai — session khatam hote hi delete ho jata hai, isliye download kar lena.

In [ ]:
# ── 5. Keep-alive (optional): backend ka status har 5 min print karta hai ────
import json, time, urllib.request
while True:
    try:
        with urllib.request.urlopen("http://127.0.0.1:3900/health", timeout=5) as r:
            print(time.strftime("%H:%M:%S"), "✅ running —", json.load(r).get("device"))
    except Exception as e:
        print(time.strftime("%H:%M:%S"), "❌ backend down:", e, "→ cell 4 dobara chalao")
        break
    time.sleep(300)